# 🧠 Consistent Identity AI — Step 1 + 2
## Environment Setup + Identity Encoder (Face & Body)

**Goal:** Given a reference photo of a person, extract a stable identity embedding that we'll later use to keep the face and body consistent across different outfits and poses.

**Pipeline position:**
```
[Reference Image] → [Identity Encoder] → [Identity Tokens] → (used in Step 3+)
```

**What this notebook covers:**
- ✅ Step 1: Install all required libraries
- ✅ Step 2A: Face embedding with InsightFace (ArcFace backbone)
- ✅ Step 2B: Body embedding with DINOv2
- ✅ Step 2C: Fuse face + body into a single identity vector
- ✅ Step 2D: Visualize and test with a sample image

> ⚡ Make sure you have **GPU enabled**: Runtime → Change runtime type → T4 GPU

---
## 📦 Step 1 — Install Dependencies

In [ ]:
# ── Core ML libraries ──────────────────────────────────────────────────────────
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate diffusers

# ── Face detection + recognition ──────────────────────────────────────────────
!pip install -q insightface
!pip install -q onnxruntime-gpu   # GPU-accelerated face model inference

# ── Image utilities ───────────────────────────────────────────────────────────
!pip install -q Pillow opencv-python-headless matplotlib

# ── Utility ───────────────────────────────────────────────────────────────────
!pip install -q numpy scipy einops

print('✅ All packages installed.')

---
## 🔧 Step 1B — Verify GPU + Imports

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import urllib.request

# ── Check GPU ─────────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected — switch to GPU runtime for best performance.')

---
## 🧑 Step 2A — Face Identity Encoder (InsightFace / ArcFace)

InsightFace detects the face region and extracts a **512-dim ArcFace embedding**.
This vector is unique to a person — similar to a fingerprint — and is very stable
across different lighting, angles, and expressions.

In [ ]:
import insightface
from insightface.app import FaceAnalysis

class FaceIdentityEncoder:
    """
    Extracts a 512-dimensional face identity embedding from an image.
    Uses InsightFace's ArcFace model (buffalo_l checkpoint).

    The embedding captures:
    - Facial geometry (eye spacing, jaw shape, nose bridge)
    - Skin texture patterns
    - Identity-specific features robust to lighting/pose changes
    """

    def __init__(self, model_pack='buffalo_l', device='cuda'):
        providers = ['CUDAExecutionProvider'] if device == 'cuda' else ['CPUExecutionProvider']
        self.app = FaceAnalysis(
            name=model_pack,
            providers=providers
        )
        # det_size: detection resolution — higher = finds smaller faces
        self.app.prepare(ctx_id=0 if device == 'cuda' else -1, det_size=(640, 640))
        print(f'✅ FaceIdentityEncoder loaded ({model_pack})')

    def encode(self, image: Image.Image) -> np.ndarray | None:
        """
        Args:
            image: PIL Image (RGB)
        Returns:
            numpy array of shape (512,) — the face identity embedding
            None if no face is detected
        """
        # InsightFace expects BGR numpy array
        img_bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

        faces = self.app.get(img_bgr)

        if len(faces) == 0:
            print('⚠️  No face detected in image.')
            return None

        if len(faces) > 1:
            print(f'ℹ️  {len(faces)} faces detected — using the largest one.')

        # Pick the face with the largest bounding box (most prominent)
        face = sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]), reverse=True)[0]

        # embedding is already L2-normalized by InsightFace
        embedding = face.normed_embedding  # shape: (512,)
        return embedding

    def encode_with_crop(self, image: Image.Image):
        """
        Returns both the embedding AND the cropped face image.
        Useful for visualization.
        """
        img_bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        faces = self.app.get(img_bgr)

        if len(faces) == 0:
            return None, None

        face = sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]), reverse=True)[0]

        # Crop the face region with 20% padding
        x1, y1, x2, y2 = [int(v) for v in face.bbox]
        pad_x = int((x2 - x1) * 0.2)
        pad_y = int((y2 - y1) * 0.2)
        h, w = img_bgr.shape[:2]
        x1 = max(0, x1 - pad_x)
        y1 = max(0, y1 - pad_y)
        x2 = min(w, x2 + pad_x)
        y2 = min(h, y2 + pad_y)

        face_crop_rgb = cv2.cvtColor(img_bgr[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
        face_crop_pil = Image.fromarray(face_crop_rgb)

        return face.normed_embedding, face_crop_pil


# ── Instantiate the encoder ────────────────────────────────────────────────────
face_encoder = FaceIdentityEncoder(device=device)

---
## 🏃 Step 2B — Body Identity Encoder (DINOv2)

DINOv2 is a self-supervised vision transformer that produces rich spatial features.
We use it to capture **body-level identity signals** — height, proportions, skin tone,
hair style — that the face encoder misses.

In [ ]:
from transformers import AutoImageProcessor, AutoModel
import torch.nn as nn

class BodyIdentityEncoder:
    """
    Extracts body-level identity features using DINOv2 (ViT-B/14).

    We take the [CLS] token output — a 768-dim global image descriptor —
    which captures body shape, proportions, hair, and skin tone.

    The image should ideally show the full body or at least torso+up.
    """

    def __init__(self, model_name='facebook/dinov2-base', device='cuda'):
        self.device = device
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()  # disable dropout for consistent embeddings
        print(f'✅ BodyIdentityEncoder loaded ({model_name})')

    @torch.no_grad()
    def encode(self, image: Image.Image) -> np.ndarray:
        """
        Args:
            image: PIL Image (RGB) — full body or upper body preferred
        Returns:
            numpy array of shape (768,) — L2-normalized body embedding
        """
        inputs = self.processor(images=image, return_tensors='pt').to(self.device)

        outputs = self.model(**inputs)

        # [CLS] token = global image summary, shape: (1, 768)
        cls_embedding = outputs.last_hidden_state[:, 0, :]

        # L2 normalize so dot product = cosine similarity
        cls_embedding = nn.functional.normalize(cls_embedding, dim=-1)

        return cls_embedding.squeeze().cpu().numpy()  # shape: (768,)


# ── Instantiate ───────────────────────────────────────────────────────────────
body_encoder = BodyIdentityEncoder(device=device)

---
## 🔀 Step 2C — Identity Fusion Module

We combine the face embedding (512-dim) and body embedding (768-dim) into a single
**fused identity vector** using a small learned projection network.

The projection maps both to a common 512-dim space, then concatenates and projects
to the final identity embedding that the diffusion model will consume.

In [ ]:
class IdentityFusionModule(nn.Module):
    """
    Fuses face and body embeddings into a single identity representation.

    Architecture:
        face  (512) ─┐
                     ├─ concat (1280) ─ LayerNorm ─ Linear ─ GELU ─ Linear ─ identity (512)
        body  (768) ─┘

    In later steps, this 512-dim vector will be projected into
    IP-Adapter identity tokens for the diffusion model.
    """

    def __init__(self, face_dim=512, body_dim=768, output_dim=512):
        super().__init__()
        fused_dim = face_dim + body_dim  # 1280

        self.face_proj = nn.Sequential(
            nn.Linear(face_dim, face_dim),
            nn.LayerNorm(face_dim),
        )
        self.body_proj = nn.Sequential(
            nn.Linear(body_dim, body_dim),
            nn.LayerNorm(body_dim),
        )
        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, fused_dim // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(fused_dim // 2, output_dim),
            nn.LayerNorm(output_dim),
        )

    def forward(
        self,
        face_emb: torch.Tensor,   # (B, 512) or (512,)
        body_emb: torch.Tensor,   # (B, 768) or (768,)
    ) -> torch.Tensor:            # (B, 512)

        # Handle unbatched inputs
        if face_emb.dim() == 1:
            face_emb = face_emb.unsqueeze(0)
        if body_emb.dim() == 1:
            body_emb = body_emb.unsqueeze(0)

        face_feat = self.face_proj(face_emb)   # (B, 512)
        body_feat = self.body_proj(body_emb)   # (B, 768)

        combined = torch.cat([face_feat, body_feat], dim=-1)  # (B, 1280)
        identity = self.fusion(combined)                       # (B, 512)

        # Final L2 normalize for stable cosine comparisons
        identity = nn.functional.normalize(identity, dim=-1)
        return identity


# ── Instantiate (randomly initialized — will be fine-tuned in later steps) ─────
fusion_module = IdentityFusionModule().to(device)
fusion_module.eval()
print('✅ IdentityFusionModule ready')
print(f'   Parameters: {sum(p.numel() for p in fusion_module.parameters()):,}')

---
## 🎯 Step 2D — Full Identity Encoder Pipeline

Wrap everything into one clean class that takes an image and returns the identity embedding.

In [ ]:
class IdentityEncoder:
    """
    Full identity encoder pipeline.

    Input  : PIL Image of a person
    Output : 512-dim identity embedding (numpy array)

    Internally runs:
        1. Face encoder  (InsightFace ArcFace) → 512-dim face embedding
        2. Body encoder  (DINOv2)              → 768-dim body embedding
        3. Fusion module                        → 512-dim fused identity
    """

    def __init__(self, face_enc, body_enc, fusion, device='cuda'):
        self.face_enc = face_enc
        self.body_enc = body_enc
        self.fusion   = fusion
        self.device   = device

    def encode(self, image: Image.Image) -> dict:
        """
        Returns a dict with:
            'identity_embedding' : np.ndarray (512,)  — final fused vector
            'face_embedding'     : np.ndarray (512,)  — face only
            'body_embedding'     : np.ndarray (768,)  — body only
            'face_crop'          : PIL.Image           — detected face crop
            'face_detected'      : bool
        """
        # ── 1. Face embedding ─────────────────────────────────────────────────
        face_emb, face_crop = self.face_enc.encode_with_crop(image)
        face_detected = face_emb is not None

        if not face_detected:
            # Fallback: use a zero vector if no face detected
            # This still lets body features drive the identity
            print('⚠️  No face found — identity will be body-only.')
            face_emb = np.zeros(512, dtype=np.float32)

        # ── 2. Body embedding ─────────────────────────────────────────────────
        body_emb = self.body_enc.encode(image)  # (768,)

        # ── 3. Fuse ───────────────────────────────────────────────────────────
        face_t = torch.tensor(face_emb, dtype=torch.float32).to(self.device)
        body_t = torch.tensor(body_emb, dtype=torch.float32).to(self.device)

        with torch.no_grad():
            fused = self.fusion(face_t, body_t)  # (1, 512)

        fused_np = fused.squeeze().cpu().numpy()

        return {
            'identity_embedding': fused_np,
            'face_embedding'    : face_emb,
            'body_embedding'    : body_emb,
            'face_crop'         : face_crop,
            'face_detected'     : face_detected,
        }

    def similarity(self, emb_a: np.ndarray, emb_b: np.ndarray) -> float:
        """
        Cosine similarity between two identity embeddings.
        1.0 = same person, 0.0 = unrelated.
        In practice >0.6 is a strong match.
        """
        return float(np.dot(emb_a, emb_b) /
                     (np.linalg.norm(emb_a) * np.linalg.norm(emb_b) + 1e-8))

    def save(self, result: dict, path: str):
        """Save identity embeddings to a .npz file for reuse."""
        np.savez(
            path,
            identity_embedding=result['identity_embedding'],
            face_embedding=result['face_embedding'],
            body_embedding=result['body_embedding'],
        )
        print(f'💾 Saved identity to {path}.npz')

    @staticmethod
    def load(path: str) -> dict:
        """Load previously saved identity embeddings."""
        data = np.load(path)
        return {
            'identity_embedding': data['identity_embedding'],
            'face_embedding'    : data['face_embedding'],
            'body_embedding'    : data['body_embedding'],
        }


# ── Assemble the full pipeline ─────────────────────────────────────────────────
identity_encoder = IdentityEncoder(
    face_enc=face_encoder,
    body_enc=body_encoder,
    fusion=fusion_module,
    device=device
)

print('✅ Full IdentityEncoder pipeline ready!')

---
## 🖼️ Step 2E — Test with a Sample Image

Upload your own photo OR run with the sample download below.

In [ ]:
# ── Option A: Upload your own image ───────────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# image_path = list(uploaded.keys())[0]
# ref_image = Image.open(image_path).convert('RGB')

# ── Option B: Use a public domain sample (Wikimedia) ──────────────────────────
sample_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/800px-Gatto_europeo4.jpg'
# ⚠️  Replace with an actual person photo for real results!
# For testing, use any portrait photo URL or upload via Option A above.

# ── Placeholder: Create a synthetic test image ────────────────────────────────
# (In real use, load an actual person photo)
test_img = Image.new('RGB', (512, 512), color=(200, 180, 160))

print('ℹ️  To test with a real person, uncomment Option A and upload a portrait photo.')
print('   The face encoder requires an actual human face to produce meaningful embeddings.')

In [ ]:
# ── Run the identity encoder ───────────────────────────────────────────────────
# Replace `test_img` with your actual portrait PIL Image
result = identity_encoder.encode(test_img)

# ── Print summary ─────────────────────────────────────────────────────────────
print('\n── Identity Encoding Results ──────────────────────────────')
print(f'Face detected       : {result["face_detected"]}')
print(f'Face embedding shape: {result["face_embedding"].shape}')    # (512,)
print(f'Body embedding shape: {result["body_embedding"].shape}')    # (768,)
print(f'Fused identity shape: {result["identity_embedding"].shape}') # (512,)
print(f'Embedding norm      : {np.linalg.norm(result["identity_embedding"]):.4f}') # should be ~1.0

In [ ]:
# ── Visualize the embedding ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Identity Embedding Visualization', fontsize=14, fontweight='bold')

# Plot 1: Original image
axes[0].imshow(test_img)
axes[0].set_title('Input image')
axes[0].axis('off')

# Plot 2: Face crop (if detected)
if result['face_crop'] is not None:
    axes[1].imshow(result['face_crop'])
    axes[1].set_title('Detected face crop')
else:
    axes[1].text(0.5, 0.5, 'No face\ndetected', ha='center', va='center', fontsize=12)
    axes[1].set_title('Face crop')
axes[1].axis('off')

# Plot 3: Identity embedding heatmap
emb = result['identity_embedding'].reshape(32, 16)  # reshape 512 → 32×16 grid
im = axes[2].imshow(emb, cmap='RdBu_r', aspect='auto', vmin=-0.15, vmax=0.15)
axes[2].set_title('Identity embedding (512-dim)')
axes[2].set_xlabel('dim →')
axes[2].set_ylabel('dim →')
plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.savefig('identity_result.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Visualization saved to identity_result.png')

In [ ]:
# ── Test: Identity similarity between two encodings of the same image ──────────
# In real use, pass two different photos of the SAME person — score should be >0.6
# Pass two different people — score should be <0.4

result_a = identity_encoder.encode(test_img)
result_b = identity_encoder.encode(test_img)  # same image → should be ~1.0

sim = identity_encoder.similarity(
    result_a['identity_embedding'],
    result_b['identity_embedding']
)
print(f'Similarity (same image, two runs): {sim:.4f}')  # should be ~1.0
print()
print('Interpretation:')
print('  > 0.7 → very likely same person')
print('  0.4–0.7 → uncertain')
print('  < 0.4 → different person')

In [ ]:
# ── Save the identity embedding for use in Step 3 ─────────────────────────────
identity_encoder.save(result, 'my_identity')

# Later steps will load this with:
# saved = IdentityEncoder.load('my_identity.npz')

print('\n✅ Step 2 complete!')
print('   Your identity embedding is saved as my_identity.npz')
print('   Next → Step 3: ControlNet pose control')

---
## 📋 Summary — What we built

| Component | Model | Output dim | Purpose |
|-----------|-------|-----------|--------|
| `FaceIdentityEncoder` | InsightFace ArcFace (buffalo_l) | 512 | Face identity fingerprint |
| `BodyIdentityEncoder` | DINOv2 ViT-B/14 | 768 | Body shape, skin, hair |
| `IdentityFusionModule` | 2-layer MLP | 512 | Merged identity token |
| `IdentityEncoder` | Full pipeline | 512 | One-stop encode + save |

## ➡️ What's next

- **Step 3** — Pose control with ControlNet + DWPose skeleton extraction
- **Step 4** — Garment encoder (outfit reference image → texture features)
- **Step 5** — Inject identity tokens into SDXL via IP-Adapter
- **Step 6** — Full inference pipeline (reference photo + prompt + outfit → output)

> 💡 The `IdentityFusionModule` is randomly initialized right now. It will be fine-tuned in Step 5 with a perceptual identity loss against real paired data.